# Nacht Garden — AMD Cloud Animation Lab

Interpolate original Peashooter tracks and composite original sprite layers using PyTorch/ROCm. The final cell reconstructs the face/eyelid attachment and exports the accepted 125-frame idle.

Place your locally extracted `peashooter-source.zip` beside this notebook. Run all cells on a ROCm-enabled AMD GPU. Outputs contain game artwork and stay private. This is cloud asset processing, not FSR or gameplay frame generation. See `../evidence/README.md` for the measured benchmark and its limits.


In [ ]:
import os, json, platform, torch
from pathlib import Path
assert torch.version.hip, "A ROCm PyTorch build is required for the AMD benchmark."
assert torch.cuda.is_available(), "ROCm GPU is not available."
# PyTorch retains the cuda namespace for HIP/ROCm devices.
GPU = torch.cuda.get_device_properties(0)
torch.set_num_threads(min(8, os.cpu_count() or 1))
ENV = dict(os=platform.platform(), pytorch=torch.__version__, rocm=torch.version.hip,
           gpu=GPU.name, vram_gib=GPU.total_memory/2**30, cpu_threads=torch.get_num_threads())
print(json.dumps(ENV, indent=2))
assert Path("peashooter-source.zip").exists(), "Upload the provided source ZIP beside this notebook."


## Renderer and original-track interpolation
Only the integration and processing code below is new. Original artwork and source tracks are verified by SHA-256.

In [ ]:
"""ROCm cloud resampling of original PvZ tracks, not FSR or AI-generated art."""
import io, json, math, time, statistics, hashlib, zipfile, platform
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F

SIZE = 512
SOURCE_FPS = 12
FIRST, COUNT = 79, 25
DURATION = COUNT / SOURCE_FPS
NUMERIC = ('x','y','sx','sy','kx','ky','a')

def load_source(path='peashooter-source.zip'):
    with zipfile.ZipFile(path) as z:
        manifest = json.loads(z.read('provenance.json'))
        for name, digest in manifest['files'].items():
            assert hashlib.sha256(z.read(name)).hexdigest() == digest, name
        root = ET.fromstring('<root>'+z.read('reanim/PeaShooterSingle.reanim').decode()+'</root>')
        tracks, pictures = [], {}
        for t in root.findall('track'):
            state = dict(x=0., y=0., sx=1., sy=1., kx=0., ky=0., a=1., f=0., i='')
            states = []
            for item in t.findall('t'):
                for e in item:
                    if e.tag in state: state[e.tag] = e.text if e.tag == 'i' else float(e.text)
                states.append(state.copy())
                if state['i'] and state['i'] not in pictures:
                    name='reanim/'+state['i'].removeprefix('IMAGE_REANIM_')+'.png'
                    pictures[state['i']] = np.array(Image.open(io.BytesIO(z.read(name))).convert('RGBA'))
            tracks.append((t.findtext('name'), states))
    return tracks, pictures, manifest

def sample(states, phase):
    """Tween numeric transforms; hold sprite identity/visibility until its keyframe."""
    i=int(math.floor(phase)) % COUNT
    a=states[min(FIRST+i,len(states)-1)]
    b=states[min(FIRST+(i+1)%COUNT,len(states)-1)]
    f=phase-math.floor(phase)
    out=a.copy()
    # Do not morph or fade between unrelated images or visibility changes.
    if a['i']==b['i'] and (a['f']>=0)==(b['f']>=0):
        for k in NUMERIC:
            delta=b[k]-a[k]
            if k in ('kx','ky'): delta=(delta+180)%360-180
            out[k]=a[k]+f*delta
    return out

class Renderer:
    def __init__(self, tracks, pictures, device):
        self.tracks=tracks; self.device=torch.device(device)
        self.images={}
        for key,a in pictures.items():
            t=torch.tensor(a.astype(np.float32)/255,device=self.device).permute(2,0,1)
            t[:3]*=t[3:4]  # Premultiplied alpha prevents dark transparent-edge halos.
            self.images[key]=t.unsqueeze(0)
        xy=torch.arange(SIZE,device=self.device,dtype=torch.float32)+0.5
        self.yy,self.xx=torch.meshgrid(xy,xy,indexing='ij')

    @torch.inference_mode()
    def render(self, phases):
        n=len(phases)
        canvas=torch.zeros((n,4,SIZE,SIZE),device=self.device)
        for name,states in self.tracks:
            sampled=[sample(states,float(p)) for p in phases]
            keys=sorted({s['i'] for s in sampled if s['i'] and s['f']>=0 and s['a']>0})
            for key in keys:
                ids=[j for j,s in enumerate(sampled) if s['i']==key and s['f']>=0 and s['a']>0]
                values=torch.tensor([[sampled[j][k] for k in NUMERIC] for j in ids],device=self.device)
                x,y,sx,sy,kx,ky,alpha=values.unbind(1)
                kx=kx*math.pi/180;ky=ky*math.pi/180
                a=torch.cos(kx)*sx;b=-torch.sin(ky)*sy;c=torch.sin(kx)*sx;d=torch.cos(ky)*sy
                det=a*d-b*c
                valid=det.abs()>1e-8;det=torch.where(valid,det,torch.ones_like(det))
                dx=self.xx[None]-x[:,None,None]-192
                dy=self.yy[None]-y[:,None,None]-192
                u=(d[:,None,None]*dx-b[:,None,None]*dy)/det[:,None,None]
                v=(-c[:,None,None]*dx+a[:,None,None]*dy)/det[:,None,None]
                img=self.images[key];h,w=img.shape[-2:]
                grid=torch.stack((2*u/w-1,2*v/h-1),-1)
                layer=F.grid_sample(img.expand(len(ids),-1,-1,-1),grid,mode='bicubic',padding_mode='zeros',align_corners=False)
                # Clamp bicubic overshoot while retaining premultiplied-alpha invariants.
                la=layer[:,3:4].clamp(0,1)
                rgb=layer[:,:3].clamp_min(0).minimum(la)
                layer=torch.cat((rgb,la),1)*alpha[:,None,None,None].clamp(0,1)*valid[:,None,None,None]
                index=torch.tensor(ids,device=self.device)
                old=canvas[index]
                canvas[index]=layer+old*(1-layer[:,3:4])
        return canvas

def sync(device):
    if torch.device(device).type=='cuda': torch.cuda.synchronize(device)

def rgba(canvas):
    alpha=canvas[:,3:4]
    rgb=torch.where(alpha>1e-7,canvas[:,:3]/alpha.clamp_min(1e-7),torch.zeros_like(canvas[:,:3]))
    return (torch.cat((rgb,alpha),1).clamp(0,1)*255).round().to(torch.uint8).permute(0,2,3,1).cpu().numpy()

def validate(cpu,gpu):
    # Original keyframes survive resampling. No new paths/art/pose design.
    for _,states in cpu.tracks:
        for i in range(COUNT):
            s=sample(states,i);original=states[min(FIRST+i,len(states)-1)]
            assert all(abs(s[k]-original[k])<1e-7 for k in NUMERIC)
            assert s['i']==original['i'] and s['f']==original['f']
    phases=[0.,0.2,0.5,5.,12.5,24.8]
    a=cpu.render(phases).cpu();b=gpu.render(phases).cpu()
    diff=(a-b).abs()
    report={'max_premultiplied_pixel_error':float(diff.max()),'mean_premultiplied_pixel_error':float(diff.mean()),'integer_keyframes_preserved':True}
    # Allow small floating-point hardware differences, not visible shape changes.
    assert report['max_premultiplied_pixel_error']<0.01,report
    assert report['mean_premultiplied_pixel_error']<0.0001,report
    return report

def benchmark(renderer,phases,repeats=3):
    renderer.render(phases);sync(renderer.device)
    times=[]
    for _ in range(repeats):
        sync(renderer.device);start=time.perf_counter()
        result=renderer.render(phases);sync(renderer.device)
        times.append(time.perf_counter()-start)
        del result
    return {'median_seconds':statistics.median(times),'trials_seconds':times,'frames_per_second':len(phases)/statistics.median(times)}

def export_clips(renderer,out='animation-results'):
    out=Path(out);out.mkdir(exist_ok=True)
    frames_by_rate={};report={}
    for fps in (12,24,60):
        n=round(DURATION*fps);assert abs(n/fps-DURATION)<1e-9
        frames=[];sync(renderer.device);start=time.perf_counter()
        for j in range(0,n,8):
            phases=np.arange(j,min(j+8,n))*SOURCE_FPS/fps
            frames.extend(Image.fromarray(a) for a in rgba(renderer.render(phases)))
        sync(renderer.device)
        boxes=[im.getbbox() for im in frames]
        assert all(boxes),'Empty frame'
        frames_by_rate[fps]=frames
        report[str(fps)]={'frames':n,'duration_seconds':n/fps,'render_and_readback_seconds':time.perf_counter()-start}
    # Common crop for all versions prevents size/anchor changes between previews.
    boxes=[im.getbbox() for fs in frames_by_rate.values() for im in fs]
    bounds=(min(b[0] for b in boxes)-2,min(b[1] for b in boxes)-2,max(b[2] for b in boxes)+2,max(b[3] for b in boxes)+2)
    for fps,frames in frames_by_rate.items():
        frames=[im.crop(bounds) for im in frames]
        # Lossless WebP preview uses millisecond intervals with cumulative rounding.
        durations=[round((i+1)*1000/fps)-round(i*1000/fps) for i in range(len(frames))]
        frames[0].save(out/f'peashooter-{fps}fps.webp',save_all=True,append_images=frames[1:],duration=durations,loop=0,lossless=True)
        w=1<<(frames[0].width-1).bit_length();h=1<<(frames[0].height-1).bit_length()
        columns=8;rows=1<<(math.ceil(len(frames)/columns)-1).bit_length()
        atlas=Image.new('RGBA',(w*columns,h*rows))
        for i,im in enumerate(frames):atlas.alpha_composite(im,((i%columns)*w+(w-im.width)//2,(i//columns)*h+h-im.height))
        atlas.save(out/f'peashooter-{fps}fps-atlas.png')
        report[str(fps)].update({'atlas_size':list(atlas.size),'uncompressed_texture_bytes':atlas.width*atlas.height*4,'common_source_bounds':bounds,'preview_duration_ms':sum(durations)})
    return report


In [ ]:
tracks, pictures, provenance = load_source()
cpu = Renderer(tracks, pictures, "cpu")
gpu = Renderer(tracks, pictures, "cuda")
quality = validate(cpu, gpu)
print("Source files verified:", len(provenance["files"]))
print(json.dumps(quality, indent=2))


## Measured CPU/GPU benchmark
Same renderer, phase samples and pixel dimensions on each device. Source textures are already resident. Timings include Python dispatch and tensor rendering, exclude initial upload, output readback, encoding and networking, and synchronize GPU completion. Both devices get a warm-up. Three trials and medians are reported; no speedup is assumed.

In [ ]:
phases = [i * 0.5 for i in range(16)]
measurements = {"cpu": benchmark(cpu, phases), "radeon_rocm": benchmark(gpu, phases)}
measurements["speedup"] = measurements["cpu"]["median_seconds"] / measurements["radeon_rocm"]["median_seconds"]
print(json.dumps(measurements, indent=2))


## Render lossless previews and atlases
24 and 60 FPS are exact multiples of the existing 12 FPS timeline, so every original keyframe remains on a sample and cycle duration is unchanged. These are animation sample rates, **not measured game FPS**. Extra frames increase local texture memory even though cloud processing has ample VRAM.

In [ ]:
texture_report = export_clips(gpu)
report = {"environment": ENV, "source": provenance, "quality": quality,
          "benchmark": measurements, "animation_outputs": texture_report,
          "game_runtime_verified": False, "original_renderer_visual_match_verified": False,
          "technology": "PyTorch ROCm transform interpolation and sprite compositing; not FSR"}
Path("animation-results/report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(texture_report, indent=2))


In [ ]:
from IPython.display import display, HTML
import base64
panels = []
for fps in (12,24,60):
    data=base64.b64encode(Path(f"animation-results/peashooter-{fps}fps.webp").read_bytes()).decode()
    panels.append(f'<div style="display:inline-block;margin:12px;text-align:center"><b>{fps} samples/sec</b><br><img style="background:#777" src="data:image/webp;base64,{data}"></div>')
display(HTML("".join(panels)))


## Final in-game integration

The next cell exports the final continuous body cycle and attached original eyelids. The recorded Nacht comparison was visually accepted by the developer. A fresh run still needs visual and engine verification; a cloud export does not automatically prove gameplay compatibility.


In [ ]:
# Keep the 25-pose body cycle continuous while the independent eyelid track blinks.
# Original body poses and all three original blink states are retained.
# No extra body poses are appended or replayed after the blink.
import copy, zipfile
from IPython.display import display, FileLink, HTML
assert torch.cuda.is_available() and torch.version.hip
original_tracks, pictures, source_manifest = load_source('peashooter-source.zip')
repair_tracks = []
for name, states in original_tracks:
    poses = [copy.deepcopy(states[min(79+i,len(states)-1)]) for i in range(25)]
    if name=='anim_blink':
        for i in range(3):poses[i]=copy.deepcopy(states[1+i])
    repair_tracks.append((name, copy.deepcopy(states[:79])+poses))
# Attach the original eyelid image to the original moving face transform.
# The head-idle layer starts at source frame 29, its attachment bind pose.
old_sampler = sample
blink_states = dict(repair_tracks)['anim_blink']
face_states = dict(repair_tracks)['anim_face']
bind_face = dict(original_tracks)['anim_face'][29]
def attachment_matrix(s):
    kx,ky=math.radians(s['kx']),math.radians(s['ky'])
    return np.array([[math.cos(kx)*s['sx'],-math.sin(ky)*s['sy'],s['x']],
                     [math.sin(kx)*s['sx'],math.cos(ky)*s['sy'],s['y']],[0.,0.,1.]])
bind_inverse=np.linalg.inv(attachment_matrix(bind_face))
attachment_errors=[]
def attached_sample(states,phase):
    pose=old_sampler(states,phase)
    if states is blink_states and pose['f']>=0:
        parent=attachment_matrix(old_sampler(face_states,phase))
        original=attachment_matrix(pose)
        matrix=parent@bind_inverse@original
        pose=pose.copy()
        pose.update(x=float(matrix[0,2]),y=float(matrix[1,2]),
                    sx=float(np.hypot(matrix[0,0],matrix[1,0])),sy=float(np.hypot(matrix[0,1],matrix[1,1])),
                    kx=math.degrees(math.atan2(matrix[1,0],matrix[0,0])),
                    ky=math.degrees(math.atan2(-matrix[0,1],matrix[1,1])))
        error=float(np.abs(np.linalg.inv(parent)@attachment_matrix(pose)-bind_inverse@original).max())
        assert error<1e-7
        attachment_errors.append(error)
    return pose
sample=attached_sample
old_count = COUNT
COUNT = 25
try:
    repaired_gpu = Renderer(repair_tracks,pictures,'cuda')
    repaired_frames=[]
    for begin in range(0,125,8):
        repaired_frames += [Image.fromarray(a) for a in rgba(repaired_gpu.render(np.arange(begin,min(begin+8,125))/5))]
    # Keyframe transforms are unchanged, including all original blink poses.
    for _,states in repair_tracks:
        if states is blink_states:continue # Its original attachment now follows the head.
        for i in range(25):
            s=sample(states,float(i))
            assert all(abs(s[k]-states[79+i][k])<1e-7 for k in NUMERIC)
    # Compare premultiplied image differences across every sample boundary.
    seam_values=[]
    for i,im in enumerate(repaired_frames):
        a=np.asarray(im).astype(np.float32)/255
        b=np.asarray(repaired_frames[(i+1)%125]).astype(np.float32)/255
        aa=a[:,:,:3]*a[:,:,3:4];bb=b[:,:,:3]*b[:,:,3:4]
        mask=(a[:,:,3]+b[:,:,3])>0.05
        seam_values.append(float(np.abs(aa-bb)[mask].mean()))
finally:
    COUNT=old_count
    sample=old_sampler
boxes=[im.getbbox() for im in repaired_frames]
bounds=(min(b[0] for b in boxes)-2,min(b[1] for b in boxes)-2,max(b[2] for b in boxes)+2,max(b[3] for b in boxes)+2)
repair_report={'gpu':torch.cuda.get_device_name(0),'rocm':torch.version.hip,'frames':125,'fps':60,'source_body_poses':25,'original_blink_states':3,'duration_msec':2083,'bounds':bounds,'last_to_first_pixel_difference':seam_values[-1],'median_step_pixel_difference':float(np.median(seam_values)),'max_step_pixel_difference':max(seam_values),'source_keyframe_transforms_preserved':True,'eyelid_face_attachment_verified':True,'max_attachment_error':max(attachment_errors),'game_runtime_verified':False}
with zipfile.ZipFile('amd-attached-eyelids.zip','w',zipfile.ZIP_DEFLATED) as out:
    for i,im in enumerate(repaired_frames):
        buffer=io.BytesIO();im.crop(bounds).save(buffer,format='PNG');out.writestr(f'frames/{i:03}.png',buffer.getvalue())
    out.writestr('report.json',json.dumps(repair_report,indent=2))
print(json.dumps(repair_report,indent=2))
preview=[im.crop(bounds) for im in repaired_frames]
durations=[round((i+1)*1000/60)-round(i*1000/60) for i in range(125)]
preview[0].save('amd-attached-eyelids.webp',save_all=True,append_images=preview[1:],duration=durations,lossless=True,loop=0)
display(HTML('<b>Continuous blink with original eyelids attached to the face — AMD ROCm, 60 samples/s</b><br><img style="background:#777" src="data:image/webp;base64,'+base64.b64encode(Path('amd-attached-eyelids.webp').read_bytes()).decode()+'">'))
display(FileLink('amd-attached-eyelids.zip'))
